## **Finetuning Qwen3 8B with Unsloth**


LLM: given tokens -> predict next token
Fine-tuning: Change the weights of how LLM predict next token by providing it examples of "this text" -> "followed by this text"

1. Load source/target pairs
2. Turn each pair into a long string which is how Qwen sees it
3. Tell trainer to only learn from brainrot answer part (second part)
4. Run training loop
5. Save adapter

In [ ]:
# 2. Get repo (Colab only)
%cd /content
!git clone -b finetune/qlora https://github.com/PrimitivePenguin/StormHack2026.git || (cd StormHack2026 && git pull)
%cd /content/StormHack2026/src_brainrot/1_LLM_finetune

In [ ]:
# 3. Install unsloth
%pip install unsloth
# %%capture

In [ ]:
# 4. Import, path, save
# import models
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
sys.path.append(str(ROOT / "src_brainrot"))

from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

from shared import SYSTEM_PROMPT, MODEL

# On Colab save to Drive to disconnect doesn't lose work
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = Path("/content/drive/MyDrive/brainrot")
except ImportError:
      SAVE = Path("./brainrot")
  
import torch

In [ ]:
# 5. Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = 256,
    load_in_4bit = True,
)

In [ ]:
# 6. attach LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 7. load dataset
ds = load_dataset("json", data_files={"train": str (ROOT / "data/train.jsonl"), "val": str(ROOT / "data/val.jsonl")})

# format: list of 3 dicts is standard format openAI, Gemini, and others use
# Given row, format a three-msg conversation
#   system: SYSTEM_PROMPT
#   user: row["source"] Ex. "Why am I like this?"
#   assistant: row["target"] Ex. "system error: brain.exe encountered feelings again"
# tokenizer.apply_chat_template() format into raw string Qwen was trained on
def format(row): # given a row, format such that
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["source"]},
            {"role": "assistant", "content": row["target"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False)}

# map the format to all rows in dataset
ds = ds.map(format)

In [ ]:
# 8. Set up training
trainer = SFTTrainer(
    model = model,                      # model to train on
    processing_class = tokenizer,       # tokenizer: text -> token ID
    train_dataset = ds["train"],    
    eval_dataset = ds["val"],           # data
    args = SFTConfig(
        dataset_text_field = "text",    # field in dataset to use for training
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        warmup_steps = 10,
        eval_strategy = "steps", eval_steps = 50,
        logging_steps = 10,
        optim="adamw_8bit",
        seed = 42,
        output_dir = str (SAVE / "outputs/checkpoints"),
        save_steps = 100,
        save_total_limit = 2,
    )
)

In [ ]:
# 9. train answers only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

print(torch.cuda.max_memory_reserved() / 1e9, "GB")

In [ ]:
# 10. Save adapter
model.save_pretrained(SAVE / "brainrot_lora")
tokenizer.save_pretrained(SAVE / "brainrot_lora")

In [ ]:
# 11. Test it minor
FastLanguageModel.for_inference(model)

for text in ["I need to finish my homework before dinner.",
             "The weather is really nice today.",
             "I'm tired, I'm going to sleep."]:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, enable_thinking=False, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=64, temperature=0.8, do_sample=True)
    print(text, "→", tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))